# 72 — Build LGBM features + train LambdaRank (Stage C)

Walks HF train conversations, splits sessions 80/20, runs the full
Stage A+B retrieval+reranker pipeline to get top-100 candidates per
music turn, then extracts the extended 28-feature vectors per
(turn, candidate) pair. Trains LightGBM LambdaRank on the result.

**Prereqs**: Stage A + Stage B done; merged BGE-M3 + CE on Hub;
BGE-M3-FT catalog pickle on Drive (notebook 70 cell 6).

**Wallclock**: ~4-6 hr on Blackwell (feature extraction is wRRF +
CE forward over ~12k music turns × 100 cands).

In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in.
# datasets/transformers import JAX transitively; JAX grabs ~75% of VRAM on
# first use, so the KERNEL ends up hogging the GPU and the cell-3 !python
# subprocess OOMs. This is why nb 72 OOM'd while nb 70/71 (which set these)
# did not. If the kernel already imported JAX, RESTART RUNTIME for this to take.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'  # G2: 3-channel union pool + new session features + album_name fix
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
src = f'{DRIVE_BASE}/recsys2026_retrieval_v2_cache'
dst = f'{LOCAL_BASE}/retrieval_v2'
if os.path.islink(dst): os.unlink(dst)
elif os.path.exists(dst):
    import shutil; shutil.rmtree(dst)
os.symlink(src, dst)

# Retrieval stack (bm25->bm25s, dense->sentence-transformers/peft) is imported
# eagerly by mcrs.retrieval_modules, so its deps are required even for the
# LGBM build. Matches nb 71's proven set + lightgbm/scikit-learn.
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' \
    'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn'

In [ ]:
# 2) Walk HF train conversations + session-disjoint 80/20 split.
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from build_bi_encoder_training_data import _iter_conversation_turns

train_conv = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
all_rows = _iter_conversation_turns(train_conv)
print(f'{len(all_rows)} per-music-turn rows from train split')
session_ids = sorted({r['session_id'] for r in all_rows})
train_sids, val_sids = train_test_split(session_ids, test_size=0.2, random_state=42)
train_set, val_set = set(train_sids), set(val_sids)
train_rows = [r for r in all_rows if r['session_id'] in train_set]
val_rows = [r for r in all_rows if r['session_id'] in val_set]
print(f'train turns: {len(train_rows)}  val turns: {len(val_rows)}')
import json as _j
os.makedirs('experiments/cache/retrieval_v2/lgbm', exist_ok=True)
with open('experiments/cache/retrieval_v2/lgbm/lgbm_train_rows.jsonl', 'w') as f:
    for r in train_rows: f.write(_j.dumps(r, default=str) + '\n')
with open('experiments/cache/retrieval_v2/lgbm/lgbm_val_rows.jsonl', 'w') as f:
    for r in val_rows: f.write(_j.dumps(r, default=str) + '\n')

In [ ]:
# 3) Extract features for each (turn, candidate) pair via Stage A+B pipeline.
# For each music turn:
#   a. Build production query via format_query_text(..., mode='bge_m3_structured').
#   b. wRRF (BM25 + dense_lyrics + BGE-M3-FT) → top-100 candidates.
#   c. CE rerank → top-100 reranked (keeps the same 100 candidates; just adds CE score).
#   d. extract_features() with the extended 28-feature vector.
# Outputs: experiments/cache/retrieval_v2/lgbm/lgbm_{train,val}_features.parquet
!cd /content/recsys2026/music-crs-baselines && python -u ../scripts/build_lgbm_features.py \
    --n-sessions 999999 \
    --topk 100 \
    --seed 42 \
    --out /content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_features.parquet \
    --cache-dir /content/recsys2026/experiments/cache \
    2>&1 | tail -20
# Note: the existing build_lgbm_features.py samples train sessions; with the
# new 80/20 split, override the sampler by writing a thin per-row driver.
# Implementation detail: pass session_ids filter via the existing --seed +
# n-sessions, OR modify build_lgbm_features.py to accept --session-id-list.
# For Phase 1, the simpler path is: run the full feature extractor on train,
# then post-filter rows to (train_sids, val_sids) into two parquets.

In [ ]:
# 4) Post-filter the single full-train parquet into 80/20 train/val by session.
import pandas as pd
df = pd.read_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_features.parquet')
tdf = df[df['session_id'].isin(train_set)].copy()
vdf = df[df['session_id'].isin(val_set)].copy()
tdf.to_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_split.parquet', index=False)
vdf.to_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_val_split.parquet', index=False)
print(f'train rows: {len(tdf)}  val rows: {len(vdf)}')
print('positives (label=1):', int(tdf['label'].sum()), int(vdf['label'].sum()))

In [ ]:
# 5) Train LightGBM LambdaRank.
!cd /content/recsys2026 && python scripts/train_lgbm_ranker.py \
    --train-features experiments/cache/retrieval_v2/lgbm/lgbm_train_split.parquet \
    --val-features   experiments/cache/retrieval_v2/lgbm/lgbm_val_split.parquet \
    --output-dir     experiments/cache/retrieval_v2/lgbm/lgbm_v1 \
    --n-estimators 1000 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm_train_log.txt
!ls -la /content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1/

In [ ]:
# 6) Copy the trained model to Drive for inference reuse.
import shutil
src = '/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1'
dst = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm/lgbm_v1'
import os
os.makedirs(os.path.dirname(dst), exist_ok=True)
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('LGBM model dir mirrored to:', dst)

In [ ]:
# 7) Offline eval (Stage A+B+C) on dev: full pipeline + LGBM rerank.
# Loads the trained LGBM via the existing LGBM_RERANKER, chains it
# after the BGE-reranker-FT in a CHAIN_RERANKER. Gate: nDCG@20 >= 0.35.
import sys, math, pickle, os
import numpy as np
from datasets import load_dataset
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.retrieval_modules.bge_m3_format import format_query_text
from build_bi_encoder_training_data import _iter_conversation_turns
from mcrs.rerankers import load_chain_reranker
from sentence_transformers import SentenceTransformer

BGE_REPO = 'OrRim123/recsys2026-bge-m3-music-v1-merged'
LGBM_DIR = '/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1'
CATALOG_PKL = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local/{BGE_REPO.replace("/","_")}/bge-m3-music-v1-merged/track_embeddings.pkl'
with open(CATALOG_PKL, 'rb') as f:
    payload = pickle.load(f)
track_ids, track_mat = payload['track_ids'], payload['track_mat']
tid_to_idx = {t: i for i, t in enumerate(track_ids)}

chain = load_chain_reranker(
    chain_spec=[
        {'type': 'bge_reranker_v2_m3', 'model_path': 'OrRim123/recsys2026-bge-reranker-music-v1', 'topk': 50},
        {'type': 'lgbm_rerank',         'model_path': LGBM_DIR,                                   'topk': 20},
    ],
    item_db_name='talkpl-ai/TalkPlayData-Challenge-Track-Metadata',
    track_split_types=['all_tracks'],
    corpus_types=['track_name', 'artist_name', 'album_name'],
    cache_dir='/content/recsys2026/experiments/cache',
)

bi = SentenceTransformer(BGE_REPO, device='cuda')
# NOTE: HF splits this dataset as 'train' and 'test'. The 'test' split IS
# the dev set for our purposes — the actual blind evaluation uses
# separate Blind-A / Blind-B datasets. All in-repo code uses split='test'.
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
rows = _iter_conversation_turns(dev)[:500]
queries = [format_query_text(r.get('chat_history') or [], r.get('current_user_query',''), r.get('user_profile_raw'), r.get('conversation_goal'), mode='bge_m3_structured') for r in rows]
q_emb = bi.encode(queries, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
q_emb = np.asarray(q_emb, dtype=np.float32)
sims = q_emb @ track_mat.T
top100_idx = np.argpartition(-sims, kth=99, axis=1)[:, :100]
ri = np.arange(sims.shape[0])[:, None]
top100_sorted = top100_idx[ri, np.argsort(-sims[ri, top100_idx], axis=1)]
cand_lists = [[track_ids[j] for j in top100_sorted[i]] for i in range(len(queries))]
out = chain.rerank(
    queries, cand_lists, topk=20,
    user_ids=[None]*len(queries), goal_categories=[None]*len(queries),
    goal_specificities=[None]*len(queries), user_profiles_raw=[None]*len(queries),
)
ndcgs = []
for i, r in enumerate(rows):
    gold = r['track_id']
    top20 = out[i]
    if gold in top20:
        rank = top20.index(gold) + 1
        ndcgs.append(1.0 / math.log2(rank + 1))
    else:
        ndcgs.append(0.0)
mean_ndcg = float(sum(ndcgs) / len(ndcgs))
print(f'Stage A+B+C nDCG@20 on dev: {mean_ndcg:.4f}  (gate: >= 0.35)')
print('PASS' if mean_ndcg >= 0.35 else 'FAIL — investigate LGBM features before Submission 3.')